In [1]:
from pathlib import Path
import sys

project_dir = Path.cwd()

required_files = [
    "README.md",
    "data/evaluation/development_v1.json",
]

if not all((project_dir / name).is_file() for name in required_files):
    raise RuntimeError("Open this notebook from the project root.")

print("Project folder:", project_dir)
print("Python executable:", sys.executable)
print("Incident and rollback lab ready.")

Project folder: C:\Users\Khaleel\workplace-ai-operations-lab
Python executable: C:\Users\Khaleel\workplace-ai-operations-lab\.venv\Scripts\python.exe
Incident and rollback lab ready.


In [2]:
import json

config_dir = project_dir / "config" / "incident_lab"
config_dir.mkdir(parents=True, exist_ok=True)

configurations = {
    "retrieval_v1.json": {
        "version": "1",
        "source_directory": "data/synthetic",
        "top_k": 2,
    },
    "retrieval_v2_invalid.json": {
        "version": "2",
        "source_directory": "data/synthetic",
        "top_k": 0,
    },
}

for filename, configuration in configurations.items():
    path = config_dir / filename
    content = json.dumps(configuration, indent=2) + "\n"

    if path.exists():
        if path.read_text(encoding="utf-8") != content:
            raise RuntimeError(f"Existing configuration differs: {filename}")
        print("Already matches:", filename)
    else:
        path.write_text(content, encoding="utf-8")
        print("Created:", filename)

print("\nRecovery acceptance criteria:")
print("- Active configuration is valid.")
print("- The leave question retrieves leave_policy:1 first.")
print("- Active configuration bytes match the known-good v1 file.")

Already matches: retrieval_v1.json
Already matches: retrieval_v2_invalid.json

Recovery acceptance criteria:
- Active configuration is valid.
- The leave question retrieves leave_policy:1 first.
- Active configuration bytes match the known-good v1 file.


In [3]:
import re


class IncidentConfigurationError(ValueError):
    """The incident-lab configuration is invalid."""


STOP_WORDS = {
    "a", "an", "the", "is", "are", "do", "does", "how",
    "what", "which", "of", "to", "for", "per", "in",
    "on", "and", "or", "with",
}


def load_validated_configuration(path):
    configuration = json.loads(path.read_text(encoding="utf-8"))

    if not isinstance(configuration, dict):
        raise IncidentConfigurationError("Configuration must be an object.")

    if not isinstance(configuration.get("version"), str):
        raise IncidentConfigurationError("A string version is required.")

    top_k = configuration.get("top_k")

    if type(top_k) is not int or not 1 <= top_k <= 10:
        raise IncidentConfigurationError(
            "top_k must be an integer from 1 through 10."
        )

    source = configuration.get("source_directory")

    if not isinstance(source, str) or not source:
        raise IncidentConfigurationError("A source directory is required.")

    relative_path = Path(source)

    if relative_path.is_absolute() or relative_path.drive or relative_path.root:
        raise IncidentConfigurationError("Source must be project-relative.")

    root = project_dir.resolve()
    source_path = (root / relative_path).resolve()

    if not source_path.is_relative_to(root) or not source_path.is_dir():
        raise IncidentConfigurationError("Source directory is invalid.")

    return configuration, source_path


def acceptance_check(configuration_path):
    try:
        configuration, source_path = load_validated_configuration(
            configuration_path
        )
    except IncidentConfigurationError as error:
        return {
            "passed": False,
            "failure": "INVALID_CONFIGURATION",
            "detail": str(error),
            "top_passage": None,
        }

    question = "How many annual leave days do full-time employees receive?"
    query_tokens = (
        set(re.findall(r"[a-z0-9]+", question.lower())) - STOP_WORDS
    )
    ranked = []

    for path in sorted(source_path.glob("*.txt")):
        text = path.read_text(encoding="utf-8")
        _, separator, body = text.partition("\n\n")

        if not separator or not body.strip():
            raise ValueError(f"Missing policy body: {path.name}")

        for line_number, line in enumerate(body.splitlines(), start=1):
            sentence = line.strip()
            if not sentence:
                continue

            tokens = (
                set(re.findall(r"[a-z0-9]+", sentence.lower()))
                - STOP_WORDS
            )
            score = len(query_tokens & tokens)

            if score:
                ranked.append({
                    "passage_id": f"{path.stem}:{line_number}",
                    "score": score,
                })

    ranked.sort(key=lambda item: (-item["score"], item["passage_id"]))
    results = ranked[:configuration["top_k"]]
    top_passage = results[0]["passage_id"] if results else None
    passed = top_passage == "leave_policy:1"

    return {
        "passed": passed,
        "failure": None if passed else "RETRIEVAL_ACCEPTANCE_FAILED",
        "detail": "Expected leave_policy:1 at rank 1.",
        "top_passage": top_passage,
    }


baseline_check = acceptance_check(config_dir / "retrieval_v1.json")
assert baseline_check["passed"], baseline_check

print("Baseline acceptance passed:", baseline_check["passed"])
print("Top passage:", baseline_check["top_passage"])

Baseline acceptance passed: True
Top passage: leave_policy:1


In [4]:
from datetime import datetime, timezone
from uuid import uuid4
import hashlib
import os

incident_id = uuid4().hex
run_dir = project_dir / "evidence" / "incident_runs" / incident_id
run_dir.mkdir(parents=True, exist_ok=False)

active_path = run_dir / "active_configuration.json"
good_bytes = (config_dir / "retrieval_v1.json").read_bytes()
bad_bytes = (config_dir / "retrieval_v2_invalid.json").read_bytes()
timeline = []


def fingerprint(content):
    return hashlib.sha256(content).hexdigest()


def record_event(action, **details):
    timeline.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "action": action,
        **details,
    })


def activate_configuration(content):
    # Write a temporary file beside the active file, then replace it.
    temporary_path = run_dir / f"pending_{uuid4().hex}.json"
    temporary_path.write_bytes(content)
    os.replace(temporary_path, active_path)


activate_configuration(good_bytes)
initial_check = acceptance_check(active_path)
assert initial_check["passed"], initial_check
record_event(
    "baseline_verified",
    configuration_sha256=fingerprint(active_path.read_bytes()),
    acceptance=initial_check,
)

# Deliberately bypass pre-activation validation for this simulation.
try:
    activate_configuration(bad_bytes)
    record_event(
        "invalid_candidate_activated",
        configuration_sha256=fingerprint(active_path.read_bytes()),
        validation_bypassed=True,
    )

    failed_check = acceptance_check(active_path)
    record_event("failure_detected", acceptance=failed_check)

    assert not failed_check["passed"], failed_check
    assert failed_check["failure"] == "INVALID_CONFIGURATION"

finally:
    # Attempt restoration even if detection or an assertion raises.
    activate_configuration(good_bytes)
    record_event(
        "rollback_applied",
        configuration_sha256=fingerprint(active_path.read_bytes()),
    )

recovery_check = acceptance_check(active_path)
bytes_restored = active_path.read_bytes() == good_bytes

record_event(
    "recovery_verified",
    acceptance=recovery_check,
    bytes_match_v1=bytes_restored,
)

assert recovery_check["passed"], recovery_check
assert bytes_restored

for event in timeline:
    print(event["action"])

print("\nDetected failure:", failed_check["failure"])
print("Recovery acceptance passed:", recovery_check["passed"])
print("Active bytes match v1:", bytes_restored)
print("Incident run folder:", run_dir)

baseline_verified
invalid_candidate_activated
failure_detected
rollback_applied
recovery_verified

Detected failure: INVALID_CONFIGURATION
Recovery acceptance passed: True
Active bytes match v1: True
Incident run folder: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\incident_runs\f6500bd6d3d442e89429a551f64b0c14


In [5]:
incident_report = {
    "incident_id": incident_id,
    "exercise_type": "Simulated configuration incident; no deployed service",
    "environment": {
        "python": sys.version.split()[0],
        "platform": sys.platform,
    },
    "change_record": {
        "baseline": "retrieval_v1.json",
        "candidate": "retrieval_v2_invalid.json",
        "changed_setting": "top_k",
        "before": 2,
        "after": 0,
        "baseline_sha256": fingerprint(good_bytes),
        "candidate_sha256": fingerprint(bad_bytes),
        "pre_activation_validation": "Deliberately bypassed",
        "rollback_trigger": "Acceptance check reports INVALID_CONFIGURATION",
    },
    "impact": (
        "The lab acceptance check rejected the active configuration "
        "before retrieval. No deployed service or users were affected."
    ),
    "cause": (
        "The candidate set top_k to 0, outside the accepted range of 1–10. "
        "The simulation activated it without pre-activation validation."
    ),
    "recovery": {
        "acceptance": recovery_check,
        "active_bytes_match_baseline": bytes_restored,
        "active_sha256": fingerprint(active_path.read_bytes()),
    },
    "timeline": timeline,
    "proposed_prevention": [
        "Validate candidate configuration before activation.",
        "Run retrieval acceptance checks before and after activation.",
        "Retain an identified known-good configuration for rollback.",
    ],
    "limitations": [
        "One acceptance query; not a full regression suite.",
        "No service restart, concurrent traffic, or live outage.",
        "No filesystem failure or rollback failure was injected.",
        "Prevention actions are proposed, not implemented by this exercise.",
        "Timeline is saved after completion, not as a durable live journal.",
    ],
}

incident_report_path = run_dir / "incident_report.json"

with incident_report_path.open("x", encoding="utf-8") as file:
    json.dump(incident_report, file, indent=2, ensure_ascii=False)

print("Incident report saved:", incident_report_path)
print("Timeline events:", len(incident_report["timeline"]))
print(
    "Verified recovery:",
    incident_report["recovery"]["acceptance"]["passed"]
    and incident_report["recovery"]["active_bytes_match_baseline"],
)

Incident report saved: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\incident_runs\f6500bd6d3d442e89429a551f64b0c14\incident_report.json
Timeline events: 5
Verified recovery: True
